In [ ]:
from pymoo.indicators.hv import HV
from scipy.stats import mannwhitneyu

import numpy as np
import pickle
import statistics
import pandas as pd
import matplotlib.pyplot as plt

# Hypervolume:

In [25]:
#################### CUSTOMIZABLE ####################

# Experiment configuration
experiment_names = ['dtlz1', 'dtlz3'] # [('zdt2', 2, 10), ('zdt4', 2, 10), ('dtlz1', 3, 10), ('dtlz3', 3, 10)]
# experiment_names = ['zdt2', 'zdt4']
objective_dim = 3
position_dim = 10

# Choosing the files
choose_global_best_attribution_type = [0] # [0, 1, 2, 3]
choose_Xr_pool_type = [0] # [0, 1, 2]
choose_DE_mutation_type = [0] # [0, 1, 2, 3, 4]

# Consider NSGA-II results
insert_nsga = False

# Create the hypervolume indicator
ref_point = np.array([10] * objective_dim)
indicator = HV(ref_point=ref_point)
######################################################

In [26]:
# Name of chosen files
file_names_mesh = [
        f"MESH_G{i+1}S{j+1}D{k+1}_{experiment_name}_{objective_dim}_{position_dim}.pkl"
        for i in choose_global_best_attribution_type
        for j in choose_Xr_pool_type
        for k in choose_DE_mutation_type
        for experiment_name in experiment_names
    ]
# Name of NSGA-II files
file_names_nsga = []
if insert_nsga:
  file_names_nsga.extend(['NSGA2_' + name + '.pkl' for name in experiment_names])

# All files
file_names = file_names_mesh + file_names_nsga
# Number of datasets
num_dataset = len(file_names)

# Take the results
results = []
results_mesh = []
results_nsga = []

for i in range(len(file_names_mesh)):
  with open("./results/" + file_names_mesh[i], "rb") as f:
    results_mesh.append(pickle.load(f))

for i in range(len(file_names_nsga)):
  with open("./results/" + file_names_nsga[i], "rb") as f:
    results_nsga.append(pickle.load(f))
results = results_mesh + results_nsga

# Create the pandas DataFrame columns
columns = ['Algorithm', 'Function', 'Mean HV', 'Std. Dev.', 'Median', 'Combined HV', 'Min. HV', 'Max. HV',]
# Store the datas
data = []
for i, fn in enumerate(file_names):
  HVs = []
  r = results[i]
  for j in range(len(results[i])-1):
    HVs.append(indicator(r[j+1]["F"]))
  exp = fn.split('_', 2)
  alg = exp[0] + ' - ' + exp[1]
  func = ((exp[2].split('_')[0]).split('.', 1))[0]
  if len(HVs) > 1:
    data.append([alg, func.upper(), statistics.mean(HVs), statistics.stdev(HVs), statistics.median(HVs), indicator(results[i]['combined'][1]), min(HVs), max(HVs),])
  else:
    data.append([alg, func.upper(), statistics.mean(HVs), 0, HVs[0], indicator(results[i]['combined'][1]), min(HVs), max(HVs),])

# Create the DataFrame with hypervolume results
df = pd.DataFrame(data, columns=columns)
df

,Algorithm,Function,Mean HV,Std. Dev.,Median,Combined HV,Min. HV,Max. HV
0,MESH - G1S1D1,DTLZ1,783.086282,217.236339,842.741425,999.039312,198.337089,999.039312
1,MESH - G1S1D1,DTLZ3,594.284775,347.204780,708.005848,982.984997,0.000000,980.833992


In [ ]:
print(df.to_latex(index=False, float_format="%.3f", escape=False, column_format='lcccccc', label='tab:hv', caption='Hypervolume indicator of the algorithms on the test problems.'))

# Teste de Hipóteses - Mann-Whitney U

In [ ]:
# Significance level
alpha = 0.05
# Store the groups for the Mann-Whitney U test
results_a = results_mesh
results_b = results_nsga

# Create the pandas DataFrame columns
columns = ['Function', 'p-value', 'Statistic', 'H0', 'Median A', 'Median B', 'Mean A', 'Mean B', 'Std. Dev. A', 'Std. Dev. B']
# Store the datas
test_data = []
for i in range(min(len(results_a), len(results_b))):
  group_a = []
  group_b = []
  for j in range(len(results[i])-1):
    group_a.append(indicator(results_a[i][j+1]["F"]))
    group_b.append(indicator(results_b[i][j+1]["F"]))
  func = experiment_names[i]
  # Perform the Mann-Whitney U test
  stat, p_valor = mannwhitneyu(group_a, group_b, alternative='two-sided')
  test_data.append([func.upper(), p_valor, stat, 'Accepts' if p_valor > alpha else 'Rejects',
                    statistics.median(group_a), statistics.median(group_b),
                    statistics.mean(group_a), statistics.mean(group_b),
                    statistics.stdev(group_a), statistics.stdev(group_b)])

# Create the DataFrame for the Mann-Whitney U test results
df_statistic = pd.DataFrame(test_data, columns=columns)
df_statistic

# Hypervolume Boxplot:

In [ ]:
# Experiment name
experiment_name = 'zdt4'
# Choosing the files
choose_global_best_attribution_type = [0] # [0, 1, 2, 3]
choose_Xr_pool_type = [0] # [0, 1, 2]
choose_DE_mutation_type = [0] # [0, 1, 2, 3, 4]
# Consider NSGA-II results
insert_nsga = False


# Name of chosen files
file_names_mesh = [
        f"MESH_G{i+1}S{j+1}D{k+1}_{experiment_name}_{objective_dim}_{position_dim}.pkl"
        for i in choose_global_best_attribution_type
        for j in choose_Xr_pool_type
        for k in choose_DE_mutation_type
    ]
# Name of NSGA-II files
file_names_nsga = []
if insert_nsga:
  file_names_nsga.extend(['NSGA2_' + experiment_name + '.pkl'])

# Take the results
results_mesh = []
results_nsga = []
for i in range(len(file_names_mesh)):
  with open("./results/" + file_names_mesh[i], "rb") as f:
    results_mesh.append(pickle.load(f))
for i in range(len(file_names_nsga)):
  with open("./results/" + file_names_nsga[i], "rb") as f:
    results_nsga.append(pickle.load(f))

# Get MESH hypervolumes
data = []
for i, fn in enumerate(file_names_mesh):
  HVs = []
  r = results_mesh[i]
  for j in range(len(results_mesh[i])-1):
    HVs.append(indicator(r[j+1]["F"]))
  data.append(HVs)
# Get NSGA-II hypervolumes if applicable
for i in range(len(file_names_nsga)):
  HVs = []
  r = results_nsga[i]
  for j in range(len(results_nsga[i])-1):
    HVs.append(indicator(r[j+1]["F"]))
  data.append(HVs)

# Creating a boxplot
labels = ['MESH - ' + fn.split('_', 1)[0] for fn in file_names_mesh] + (['NSGA-II'] if insert_nsga else [])
plt.figure(figsize=(8, 5))
plt.boxplot(data, tick_labels=labels, showmeans=False)

# Titles and labels
# plt.title('Hypervolume - ' + experiment_name.upper(), fontsize=14)
plt.xlabel('Algoritmos') # plt.xlabel('Algorithms')
plt.ylabel('Hipervolume') # plt.ylabel('Hypervolume')
plt.grid(True, axis='y', linestyle='--', alpha=0.6)
plt.show()